# HelpSteer3 DPO pipeline

This notebook runs standard Direct Preference Optimization from the frozen one-epoch SFT policy. Compute stays on the fast Colab SSD; prepared data, periodic Trainer checkpoints, final models, and evaluation artifacts are persisted to Google Drive.

Each numbered section is restartable. After a runtime reset, rerun Sections 0 and 1, then jump directly to the unfinished stage. The training command uses `resume_from_checkpoint=auto`, so restoring the DPO run directory also restores optimizer, scheduler, RNG, and Trainer state.

## 0. Runtime and repository

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/djdhillxn/rlhf.git"
REPO_ROOT = Path("/content/rlhf")
LOCAL_RUN_ROOT = Path("/content/rlhf_runs/qwen25_05b_helpsteer3_trl_a100_full/full")
DRIVE_RUN_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/8mile/rlhf_runs/qwen25_05b_helpsteer3_trl_a100_full/full")
DRIVE_LIGHTWEIGHT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/8mile/rlhf_runs_lightweight_export/qwen25_05b_helpsteer3_trl_a100_full/full")

DPO_RUN_NAME = "dpo_standard_b01_r4096"
DPO_CONFIG = "configs/trl/qwen25_05b_helpsteer3_dpo.yaml"
DPO_EVAL_CONFIG = "configs/trl/qwen25_05b_helpsteer3_dpo_eval_suite.yaml"

CACHE_DIR = LOCAL_RUN_ROOT / "data"
SFT_DIR = LOCAL_RUN_ROOT / "sft"
SFT_MODEL = SFT_DIR / "final_merged_model"
DPO_DIR = LOCAL_RUN_ROOT / DPO_RUN_NAME
DRIVE_DPO_DIR = DRIVE_RUN_ROOT / DPO_RUN_NAME
EVAL_DIR = LOCAL_RUN_ROOT / "eval_dpo_standard_full"
DRIVE_EVAL_DIR = DRIVE_RUN_ROOT / "eval_dpo_standard_full"

for path in (LOCAL_RUN_ROOT, CACHE_DIR, DPO_DIR, EVAL_DIR):
    path.mkdir(parents=True, exist_ok=True)

print("Local run root:", LOCAL_RUN_ROOT)
print("Drive run root:", DRIVE_RUN_ROOT)

In [ ]:
if not (REPO_ROOT / ".git").is_dir():
    !git clone {REPO_URL} {REPO_ROOT}

%cd {REPO_ROOT}
!git pull --ff-only
!python -m pip install -q -e ".[trl]"

%env PYTHONUNBUFFERED=1
%env TOKENIZERS_PARALLELISM=false
%env TRL_EXPERIMENTAL_SILENCE=1

In [ ]:
import torch
import transformers
import trl

assert torch.cuda.is_available(), "Select a GPU runtime before continuing."
properties = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"GPU memory: {properties.total_memory / 2**30:.1f} GiB")
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)

## 1. Restore persistent prerequisites

These cells restore only what DPO needs: the merged SFT policy, any previously prepared DPO dataset, and any interrupted DPO run. A missing DPO dataset or run is normal on the first launch.

In [ ]:
!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_RUN_ROOT / 'sft' / 'final_merged_model'}" \
    --destination "{SFT_MODEL}" \
    --profile full

!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_RUN_ROOT / 'data'}" \
    --destination "{CACHE_DIR}" \
    --profile full \
    --include dpo \
    --include preparation_report.json \
    --allow-missing

!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_DPO_DIR}" \
    --destination "{DPO_DIR}" \
    --profile full \
    --allow-missing

print("SFT ready:", (SFT_MODEL / "config.json").is_file())
print("DPO data ready:", (CACHE_DIR / "dpo" / "train" / "dataset_info.json").is_file())
print("Restored DPO checkpoints:", sorted(path.name for path in DPO_DIR.glob("checkpoint-*")))

## 2. Prepare preference data once

Run this section only when `DPO data ready` above is false. Tied pairs are excluded. The preferred response becomes `chosen`, the other becomes `rejected`, complete responses retain EOS, and old prompt turns are dropped before any token-level fallback. The main run uses each non-tied pair once; HelpSteer3 preference strengths remain metadata for stratified analysis rather than being assumed to be linearly spaced weights.

In [ ]:
!python -m scripts.rlhf_trl_prepare_data \
    --config {DPO_CONFIG} \
    --set 'model.sft_model_path="{SFT_MODEL}"' \
    --set 'data.cache_dir="{CACHE_DIR}"'

!python -m scripts.rlhf_sync_runs \
    --source "{CACHE_DIR}" \
    --destination "{DRIVE_RUN_ROOT / 'data'}" \
    --profile full \
    --include dpo \
    --include preparation_report.json \
    --include tokenizer

## 3. Preflight

In [ ]:
!python -m scripts.rlhf_trl_doctor \
    --config {DPO_CONFIG} \
    --stage dpo \
    --set 'model.sft_model_path="{SFT_MODEL}"' \
    --set 'data.cache_dir="{CACHE_DIR}"' \
    --set 'train.output_dir="{DPO_DIR}"' \
    --set 'train.checkpoint_sync_dir="{DRIVE_DPO_DIR}"' \
    --set 'train.final_sync_dir="{DRIVE_DPO_DIR}"'

## 4. Optional end-to-end check

Run this once after a code change, then delete the local smoke directory if desired. It does not write checkpoints into the full-run directory.

In [ ]:
SMOKE_DIR = LOCAL_RUN_ROOT / "dpo_smoke"

!python -m scripts.rlhf_trl_train_dpo \
    --config {DPO_CONFIG} \
    --set 'model.sft_model_path="{SFT_MODEL}"' \
    --set 'data.cache_dir="{CACHE_DIR}"' \
    --set 'data.max_train_samples=128' \
    --set 'data.max_eval_samples=32' \
    --set 'train.output_dir="{SMOKE_DIR}"' \
    --set 'train.max_steps=4' \
    --set 'train.per_device_train_batch_size=2' \
    --set 'train.per_device_eval_batch_size=2' \
    --set 'train.gradient_accumulation_steps=2' \
    --set 'train.eval_steps=2' \
    --set 'train.save_steps=2' \
    --set 'train.resume_from_checkpoint=null' \
    --set 'train.checkpoint_sync_dir=null' \
    --set 'train.final_sync_dir=null' \
    --set 'dpo.precompute_ref_batch_size=2'

## 5. Full DPO run or exact resume

The H100 80 GB profile uses micro-batch 8 and gradient accumulation 4, for an effective batch of 32. One epoch is about 1,134 optimizer steps for 36,264 non-tied training pairs. Every 200 optimizer steps, the complete Trainer checkpoint is copied to Drive. On interruption, rerun Sections 0, 1, 3, and this section; `auto` resumes from the newest restored checkpoint.

In [ ]:
!python -m scripts.rlhf_trl_train_dpo \
    --config {DPO_CONFIG} \
    --set 'model.sft_model_path="{SFT_MODEL}"' \
    --set 'data.cache_dir="{CACHE_DIR}"' \
    --set 'train.output_dir="{DPO_DIR}"' \
    --set 'train.checkpoint_sync_dir="{DRIVE_DPO_DIR}"' \
    --set 'train.final_sync_dir="{DRIVE_DPO_DIR}"' \
    --set 'train.resume_from_checkpoint=auto'

### A100 40 GB fallback

Use these four additional overrides only if the default profile does not fit: `train.per_device_train_batch_size=4`, `train.gradient_accumulation_steps=8`, `train.per_device_eval_batch_size=2`, and `dpo.precompute_ref_batch_size=4`. The effective training batch remains 32.

## 6. Restore comparison assets

Evaluation uses the epoch-two reward model only as one diagnostic judge and retains repetition, length, EOS, and qualitative audits. Restore the final reward model and the final PPO adapter before running the four-policy suite.

In [ ]:
REWARD_DIR = LOCAL_RUN_ROOT / "reward_epoch2"
PPO_RUN_NAME = "ppo_nplus_exact_eos_r512_b64_kl10_rm_ep2"
PPO_CHECKPOINT = LOCAL_RUN_ROOT / PPO_RUN_NAME / "checkpoint-100"
DRIVE_PPO_CHECKPOINT = DRIVE_RUN_ROOT / PPO_RUN_NAME / "checkpoints" / "checkpoint-100"

!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_RUN_ROOT / 'reward_epoch2'}" \
    --destination "{REWARD_DIR}" \
    --profile full \
    --include final_merged_model \
    --include reward_center.json

!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_PPO_CHECKPOINT}" \
    --destination "{PPO_CHECKPOINT}" \
    --profile full

## 7. Full validation evaluation

This generates Base, SFT, PPO, and DPO responses for all 2,017 validation prompts with the same decoding settings. The evaluator is resumable at the policy-output level, so rerunning this command continues incomplete JSONL files instead of regenerating completed policies.

In [ ]:
!python -m scripts.rlhf_evaluate_policy_suite \
    --config {DPO_EVAL_CONFIG} \
    --set 'model.tokenizer_path="{SFT_MODEL}"' \
    --set 'reward_model.checkpoint_dir="{REWARD_DIR / "final_merged_model"}"' \
    --set 'reward_model.reward_center_path="{REWARD_DIR / "reward_center.json"}"' \
    --set 'policies[1].checkpoint_dir="{SFT_MODEL}"' \
    --set 'policies[2].checkpoint_dir="{SFT_MODEL}"' \
    --set 'policies[2].base_model_path="{SFT_MODEL}"' \
    --set 'policies[2].adapter_path="{PPO_CHECKPOINT}"' \
    --set 'policies[3].checkpoint_dir="{DPO_DIR / "final_merged_model"}"' \
    --set 'eval.output_dir="{EVAL_DIR}"' \
    --set 'eval.num_prompts=all'

!python -m scripts.rlhf_sync_runs \
    --source "{EVAL_DIR}" \
    --destination "{DRIVE_EVAL_DIR}" \
    --profile full

## 8. Lightweight analysis export

This Drive copy omits tensors and files over 50 MB. After Google Drive for desktop has synchronized, run `python -m scripts.rlhf_sync_runs --profile lightweight` from the repository on the Mac. The script discovers the Drive folder and updates `rlhf_runs_lightweight_export/` with configs, Trainer state, metrics, CSV/JSON/JSONL, markdown reports, tokenizers, and plots.

In [ ]:
!python -m scripts.rlhf_sync_runs \
    --source "{DRIVE_RUN_ROOT}" \
    --destination "{DRIVE_LIGHTWEIGHT_ROOT}" \
    --profile lightweight \
    --max-size-mb 50